# Task 4 — Valutazione del classificatore 1R su `training.csv`

> 📄 Documentazione completa e motivazioni: [`docs/task4.md`](../docs/task4.md)

L'obiettivo e` quello di valutare il classificatore **1R** implementato nel Task 2 su un dataset piu ampio , cercando di **ottimizzarne le prestazioni**.

## 0. Setup

In questo primo step , vengono importate le librerie e il dataset `training.csv`.


In [11]:
import pandas as pd
import numpy as np

In [12]:
training = pd.read_csv("../data/processed/training.csv", sep=";")
dataFrame_1R = training.copy() 

## 1. Richiamo di variabili e metodi utilizzati in 1R
In questa sezione riprendiamo il modello e i metodi necessari per testare 1R su `training.csv`.
In particolare riprendiamo il **modello 1R** appreso nel Task 2 (che ha scelto l'attributo `marital`) e la funzione **predici_1R**, che applica la regola appresa a una nuova istanza.


In [13]:
# Modello 1R appreso nel Task 2 su manuale.csv: l'attributo scelto è 'marital'
modello_1R = {
    "attributo": "marital",
    "regole": {"divorced": 1, "married": 0, "single": 1},
    "soglia": None,
}

In [14]:
def predici_1R(modello, istanza):
    attr = modello["attributo"]
    if modello["soglia"] is not None:                 # attributo numerico
        val = "high" if istanza[attr] > modello["soglia"] else "low"
    else:                                             # attributo nominale
        val = istanza[attr]
    return modello["regole"].get(val, 0)              # default 0 se valore mai visto

## 2. 1R su `training.csv`

Una volta ripreso cio che ci serve , possiamo testare il modello su `training.csv`.

In [15]:
dataFrame_1R["Predicted"] = dataFrame_1R.apply(lambda row: predici_1R(modello_1R, row),axis=1)
dataFrame_1R[["y", "Predicted"]].head(10)

,y,Predicted
0,0,0
1,0,0
2,0,0
3,0,0
4,0,0
5,0,0
6,0,0
7,0,0
8,0,1
9,0,1


## 3. Valutazione delle prestazioni

Una volta applicato il modello sul dataset `training.csv`, possiamo valutare le prestazioni basandoci sui seguenti criteri :

- Confusion Matrix
- Accuracy
- Precision
- Recall
- F1

I parametri utilizzati per calcolarci tali metriche di valutazione sono :

- TP = True Positive
- TN = True Negative
- FP = False Positive
- FN = False Negative


## 3.1 Confusion Matrix

In [16]:

tp = len(dataFrame_1R[(dataFrame_1R["y"] == 1) & (dataFrame_1R["Predicted"] == 1)])
tn = len(dataFrame_1R[(dataFrame_1R["y"] == 0) & (dataFrame_1R["Predicted"] == 0)])
fp = len(dataFrame_1R[(dataFrame_1R["y"] == 0) & (dataFrame_1R["Predicted"] == 1)])
fn = len(dataFrame_1R[(dataFrame_1R["y"] == 1) & (dataFrame_1R["Predicted"] == 0)])


confusion_matrix_df = pd.DataFrame(
    [[tn, fp],
     [fn, tp]],
    columns=["Predetto 0", "Predetto 1"],
    index=["Reale 0", "Reale 1"]
)
print("TP = ", tp)
print("TN = ", tn)
print("FP = ", fp)
print("FN = ", fn)
print("\nMatrice di Confusione:")
confusion_matrix_df


TP =  2096
TN =  22458
FP =  14079
FN =  2543

Matrice di Confusione:


,Predetto 0,Predetto 1
Reale 0,22458,14079
Reale 1,2543,2096


## 3.2 Accuracy

L'**accuracy** indica quanto e stato bravo il modello.

In [17]:
accuracy = (dataFrame_1R["y"] == dataFrame_1R["Predicted"]).mean()
print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.5963


## 3.3 Precision

La **precision** indica in percentuale quanti tra tutti i positivi individuati sono realmente positivi.

$$ \text{Precision} = \frac{TP}{TP + FP} $$

In [18]:
precision = tp / (tp + fp)
print(f"Precision: {precision:.4f}")

Precision: 0.1296


## 3.4 Recall

La **recall** indica in percentuale quanti positivi sono stati individuati

$$ \text{Recall} = \frac{TP}{TP + FN} $$

In [19]:
recall = tp / (tp + fn)
print(f"Recall: {recall:.4f}")

Recall: 0.4518


## 3.5 F1

L' **F1** rappresenta la media ponderata tra precision e recall.

$$ F1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}} $$

In [20]:
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
print(f"F1 Score: {f1:.4f}")

F1 Score: 0.2014


## 4. Analisi dei risultati

Il classificatore 1R (basato sull'attributo `marital`) ha ottenuto i seguenti risultati :

| Metrica | Valore |
|---|---|
| Accuracy  | 59.63% |
| Precision | 12.96% |
| Recall    | 45.18% |
| F1-Score  | 20.14% |

con la seguente matrice di confusione:

|            | Predicted 0 | Predicted 1 |
|------------|:-----------:|:-----------:|
| **Actual 0** | 22458 | 14079 |
| **Actual 1** | 2543 | 2096 |

L'accuracy risulta moderata, ma non rappresenta da sola una misura affidabile delle prestazioni, poiché il dataset presenta un forte sbilanciamento tra le classi.

La regola `marital` (appresa sul `manuale.csv` bilanciato) assegna alla classe 1 i clienti `single` e `divorced`: sul dataset reale questo produce **moltissimi falsi positivi** (14079), da cui una precision molto bassa (12.96%).

La recall (45.18%) mostra che il modello individua meno della metà dei reali positivi, mentre l'F1-score (20.14%) conferma che il compromesso tra precision e recall è tutt'altro che ottimale.

> **Nota.** Questi valori sono **identici** a quelli ottenuti dal Naive Bayes nel notebook `task4_NaiveBayes.ipynb`. Non è un caso: nel Naive Bayes gli attributi `housing` e `loan` si erano rivelati poco discriminanti (distribuzioni quasi identiche nelle due classi), quindi la decisione era di fatto guidata dal solo `marital`, lo stesso attributo su cui poggia 1R. I due modelli, su questi dati, prendono quindi le **stesse** decisioni — coerentemente con quanto già osservato nel Task 2. La spiegazione dettagliata è riportata in fondo a `task4_NaiveBayes.ipynb`.


## 5. Conclusione: 1R non è ottimizzabile senza riaddestramento

A differenza di Naive Bayes — dove è possibile intervenire sulle **probabilità a priori** senza riaddestrare il modello (vedi `task4_NaiveBayes.ipynb`) — **1R non offre alcuna leva di ottimizzazione che non comporti un riapprendimento**.

Il "modello" 1R, infatti, è semplicemente una **regola di maggioranza** che associa a ogni valore dell'attributo scelto (`marital`) una classe: non esistono né probabilità a priori, né soglie, né iperparametri da regolare a posteriori. L'unica cosa modificabile sarebbe la **mappa valore → classe**, ma ricavarla significa **riaddestrare** la regola sui dati — operazione che qui, in coerenza con l'approccio del Task 2, abbiamo scelto di non fare. Per questo, in questo notebook, non viene presentata alcuna fase di "ottimizzazione".

Questo è di per sé un risultato significativo: la **semplicità** di 1R, che nel Task 2 era un pregio (trasparenza, nessun overfitting), qui si traduce in **rigidità**, cioè nell'assenza di margini di intervento a parità di regola appresa.

Le vere leve di ottimizzazione — **scegliere l'attributo migliore** sul dataset grande (es. `poutcome`, emerso come molto informativo nel Task 3), **discretizzare** le variabili numeriche ed escludere quelle problematiche, e soprattutto **gestire lo sbilanciamento** delle classi — richiedono tutte un riaddestramento e vengono affrontate nel **Task 5**.
